## tl;dr
The five-market V1 history replays exactly, but the Dusk calibration does not produce a certifiable winner. The only structurally executable sampled profile has 77 deferred LP withdrawals and 1,695 missing continuous price keys, so comparative headline metrics remain unavailable rather than being guessed.


## Context & Methods
This diagnostic notebook reads the compact, checksum-bound evidence bundle created from the sealed DecisionGrade run. It validates exact replay counts, cohort coverage, calibration viability, and the no-winner decision.

### Key Assumptions
- Unavailable external LP ownership state is not imputed.
- Missing hourly USD marks are not forward-filled or replaced with current prices.
- Profiles rejected by Dusk's exact `InvalidMarketConfig` curve geometry are excluded, not treated as runtime failures.


## Data
Load the compact report evidence beside this notebook.


In [1]:
import json
from pathlib import Path

evidence_path = Path('evidence/benchmark-diagnostic.json')
evidence = json.loads(evidence_path.read_text())
print(f"run={evidence['run_id']} cutoff={evidence['cutoff_slot']}")
print(f"sources={len(evidence['source_artifacts'])} checksum-bound artifacts")


run=slot-440459364-06c0e05aadb7e883 cutoff=440459364
sources=9 checksum-bound artifacts


## Results
Run exact consistency assertions and print the decision table.


In [2]:
replay = evidence['replay']
assert replay['canonical_operations'] == 139_812
assert replay['replayed_transitions'] == 139_825
assert replay['raw_unit_differences'] == 0
assert replay['conservation_failures'] == 0
assert sum(row['transitions'] for row in evidence['market_transitions']) == replay['replayed_transitions']
calibration = evidence['calibration']
assert calibration['profiles_sampled'] == calibration['geometry_rejected_profiles'] + calibration['executable_profiles']
assert calibration['certified_scored_profiles'] == 0
print('Exact replay: PASS')
print('Certified comparative winner: NONE')
print(json.dumps(calibration, indent=2))


Exact replay: PASS
Certified comparative winner: NONE
{
  "profiles_sampled": 32,
  "geometry_rejected_profiles": 31,
  "executable_profiles": 1,
  "executable_profile_id": "d36b6a5465308a79cb68c7fa815eca2ef41ad688d31cfe6610817cdf390e0d3b",
  "certified_scored_profiles": 0,
  "deferred_lp_withdrawals_per_arm": 77,
  "missing_continuous_price_keys_per_arm": 1695,
  "canonical_operations_in_training": 113735,
  "rejected_counterfactual_operations_fixed_flow": 110061,
  "rejected_counterfactual_operations_intent": 203914,
  "blocking_reasons": [
    "capital_state_missing_at_window_start",
    "counterfactual_replay_deferred_external_state",
    "fixed_flow_arm_has_no_complete_candidate_score",
    "intent_arm_has_no_complete_candidate_score",
    "missing_historical_usd_marks",
    "no_capital_denominator",
    "no_equal_market_revenue_observations"
  ]
}


In [3]:
print('Selected-panel coverage')
for row in evidence['cohort_coverage']:
    print(f"{row['activity']:<14} {row['selected_rows']:>7,}/{row['visible_rows']:<7,} ({row['coverage_pct']:.2f}%)")
print('\nFinal gate limitations')
for gate in evidence['gate_rows']:
    if gate['status'] != 'pass':
        print(f"{gate['gate']}: {gate['status']} - {gate['detail']}")


Selected-panel coverage
Swaps          135,426/472,002 (28.69%)
Debt actions     3,227/3,868   (83.42%)
Liquidations        72/99      (72.72%)

Final gate limitations
Decoder Revision Certification: fail - every historical program revision has one verified decoder
Terminal State: fail - terminal raw state matches on chain exactly


## Takeaways
1. Historical V1 accounting is trustworthy for this selected panel: every exact raw-unit and conservation check passed.
2. The panel is lending-heavy, covering 83.42% of visible debt actions and 72.72% of liquidations, but only 28.69% of swaps.
3. Dusk cannot be declared better or worse from this run. One sampled profile is structurally executable, but zero profiles have complete scoring evidence.
4. The decision-safe action is to publish a no-winner result and avoid spending more storage on a forced full calibration.


## Actual Data Catalog
List the retained row-level datasets and load representative records from the real files.


In [4]:
catalog = json.loads(Path('evidence/data-catalog.json').read_text())
for row in catalog['datasets']:
    print(f"{row['dataset']:<24} {row['rows']:>10,} rows  {row['size']:<22} {row['quality']}")
sample = json.loads(Path('data-samples/canonical-operations.sample.json').read_text())
print('\nCanonical operation sample IDs:')
for row in sample['records']:
    print(row['operation_id'], row['order'], row.get('pair'))


Canonical operations        139,812 rows  359.5 MiB + 123.3 MiB  Exact; zero normalization gaps
Canonical events            264,017 rows  260.9 MiB + 89.1 MiB   Exact chronological order
V1 replay ledger            139,825 rows  675.5 MiB + 90.7 MiB   Zero raw-unit differences; zero conservation failures
Historical prices            22,071 rows  3.9 MiB + 597.5 KiB    Historical-only; no interpolation/current fallback
Decoded transactions        137,280 rows  1.1 GiB                Zero transaction decode failures
Decoded instructions        462,312 rows  468.9 MiB              160,749/160,749 state-changing instructions decoded
Execution provenance        137,280 rows  1.4 GiB                Exact instruction/CPI ownership evidence
Database events             895,276 rows  543.1 MiB              Discovery source; reconciled against RPC
DB/RPC reconciliation       263,977 rows  74.4 MiB               263,977/263,977 pass or documented duplicate/omission
Archived proof blocks       115,